# Safeguards before real-process qualification
## A guided team review • 12 November 2026

**DECISION: real-process qualification remains gated.** This notebook documents the proposed safeguards, the read-only environment findings and the smallest planned test batch. It does not implement or execute them.

| Ready to inspect | Proposed, not implemented here | Not established |
|---|---|---|
| Existing mock-tested candidate; source review; environment metadata | Independent guardian, process containment, bounded cleanup; ten-scenario plan | Real-process qualification, native Temporal qualification, measured results |

GitHub displays this notebook as a static document. **There are no code cells to run.** No Jupyter installation is needed to read it. Actual GitHub/Jupyter rendering has not been verified.

[Research-book contents](../../notebooks/00_start_here.ipynb) · [Study report](../../RESEARCH_STUDY.ipynb) · [Experimental candidate](README.md) · [Draft PR #7](https://github.com/node-and-norm/revocation-delegated-workflows/pull/7)

**Latest safety finding:** an ordinary graceful VM-stop request may still force-stop the shared user-v2 network helper. “No force” below means no requested forced VM shutdown, not no helper SIGKILL. Read section 4i before any future operation decision.


## Choose a reading route

**Organizational leaders:** read the decision, the safety architecture, the environment findings and the next gate.

**Researchers:** add the test matrix, evidence rules, verdicts and limits.

**Engineers and reviewers:** read every section, then inspect the pinned implementation and historical failure record. Do not treat this plan as an execution instruction or a passed control.

The central question is simple: **if the exchange gets stuck, can a separate mechanism stop the whole test and show what actually happened?**


## 1. Where we are now

Baseline: PR #7 at **`f1e50910d4cdd41010840520f209f1ae084138b1`**, inspected 12 November 2026. Its 37 mock/pure test methods are recorded as passing; they were **not rerun for this notebook**. No real child process, socket, Temporal Workflow/Activity or measured study ran in that candidate qualification.

The candidate persists dispatch intent, sends once, bounds a process join, rejects same-object reuse and preserves uncertainty. These are useful mechanisms, but their tests use process/channel/clock doubles.

**A bounded join is not a bounded whole operation.** Startup, filesystem persistence, termination and other OS operations can still block. The candidate is unintegrated development code, not a replacement for the native collector.

Sources: [pinned status](https://github.com/node-and-norm/revocation-delegated-workflows/blob/f1e50910d4cdd41010840520f209f1ae084138b1/experiments/temporal-qualification-v0.1/STATUS.md), [pinned exchange](https://github.com/node-and-norm/revocation-delegated-workflows/blob/f1e50910d4cdd41010840520f209f1ae084138b1/experiments/temporal-qualification-v0.1/bounded_exchange.py), [earlier partial-qualification decision](../../docs/decisions/2026-11-04-temporal-partial-qualification.md).


## 2. The proposed emergency-stop architecture

```text
Independent guardian (outside the test boundary)
    │ one launch instruction; fixed deadline; bounded status
    ▼
Dedicated, externally managed containment boundary
    ├── trusted launcher / controller
    ├── single-use exchange worker
    └── synthetic gateway / supervised evidence writer
```

1. Establish the boundary and its ownership **before** admitting an attempt.
2. Verify boundary READY before admission: guardian/boundary identity and configured limits. Then require per-attempt READY within the setup budget, binding attempt identity and descriptor custody. Neither is simply an unverified message.
3. After boundary READY, the guardian admits one attempt and starts its absolute deadline; exchange dispatch waits for per-attempt READY. It must not block in process startup, journal writes, hashing or pipe reads.
4. A supervised launcher performs potentially blocking setup. Capped, nonblocking status messages cannot reset the deadline.
5. At failure or timeout, use a separate lifecycle mechanism to stop the entire boundary. Never send a cleanup command through a possibly partial execute channel.
6. Independently observe exit of all in-scope tasks. A successful signal request is not proof they stopped.

**Preferred:** a documented delegated OS scope/cgroup or provider-managed disposable container with an external owner. Availability here is not established. A fresh coding container alone does not prove these properties.


## 3. Proposed timing and cleanup contract

These are **uncalibrated engineering budgets**, not study observation windows. Freeze them before any approved run; do not extend a live case to obtain a pass.

| Phase | Maximum / absolute rule |
|---|---|
| Guardian/boundary bootstrap | Before admission; no test dispatch permitted |
| Attempt setup and readiness | 10 seconds from admission |
| Single exchange | 5 seconds; end no later than second 15 |
| Graceful cleanup | 5 seconds, clipped to the absolute schedule |
| Forced containment and exit observation | Remaining budget; final cutoff at second 30 |

An earlier failure begins cleanup immediately. Heartbeats never renew the budget. Failure to prepare the guardian or boundary prevents admission.

At the graceful deadline, escalate through the boundary's force-stop mechanism. At the final cutoff, unknown process membership or unresolved termination means **cleanup_unresolved**. Stop the batch and do not reuse the transport or boundary. Retain ownership for operator/provider recovery.

On normal completion, capture evidence before cleanup. On emergency timeout, stop promptly even if capture is incomplete. Post-stop evidence must not fill an earlier observation gap.

This is not a hard-real-time guarantee against kernel hangs, host suspension or guardian failure. Those require a separately verified external service and fault model.


## 4. What the cloud inspection actually showed

Read-only inspection on **12 November 2026** used OS metadata and command/path discovery. No test workload, signal, socket, namespace operation or service-manager action was attempted.

| Observed metadata | What it supports | What it does not establish |
|---|---|---|
| Linux 6.18.57 x86_64; unprivileged user | Identity of the inspected runtime | Behavior of another/future environment |
| Capability masks all zero; NoNewPrivs=1; Seccomp=2 | Restricted current process context | Exact syscall policy or proof every operation is forbidden |
| `/proc/self/cgroup`: `0::/`; no visible cgroup mount; `/sys/fs/cgroup` absent | No exposed delegated cgroup control found | Host has no cgroups |
| PID 1: `codex`; ordinary systemd/D-Bus endpoints absent | No evidenced service-manager control route | All platform management is absent |
| `systemctl`, `systemd-run`, `timeout`, `nsenter`, `unshare` on PATH | Binaries are installed | Permission or verified containment semantics |
| Docker/Podman commands not found on PATH; usual sockets absent | No discovered local container-control route | No provider-managed containers exist |

**Conclusion:** the inspected environment does not expose the stronger controls needed by this design through the paths examined. Provider-level capabilities remain unknown, not disproved.

See [inspection evidence](safeguards/RUNTIME_INSPECTION.md) for exact commands and selected outputs. This is a transcribed record of observed tool output, not an independently signed host attestation or a runtime control test.

### Mac candidate: stopped status, not startup clearance

A separate read-only environment review reports that **both Colima and Lima list `node-norm-research` as stopped**. These are related management views, not independent proof of complete process absence or verified containment/stop semantics.

The saved profile includes **persistent disks, user-mode networking, host-name resolution, proxy propagation and broad automatic forwarding**. Existing workloads and restart policies have not been inspected. Starting the VM could resume retained services or workloads and expose network paths before a qualification fixture is ready; stopped status does not make startup inert.

A Rosetta-related `sysctl` read was denied and was **not retried**. No denial was bypassed. This chapter does not establish Rosetta/runtime suitability.

**Do not start this VM under the present documentation step.** Before a separate startup decision, review retained workload/restart behavior and the concrete network exposure, then establish the independent watchdog/containment controls. No security-setting change is implied. See the inspection record for attribution and limits.


## 4a. Recovered history: the October 2 sandbox

The earlier repository name redirects; the current source is **control-evidence-corpus**, pinned here to commit `f3e8066e2f1db268255914ed0bde08bcd415002b`. This is a **different deterministic fixture**, not the current Temporal/revocation transport candidate.

| Recorded historical evidence | Limit on reuse |
|---|---|
| October 2 local date (some timestamps October 3 UTC): 8 effective-configuration checks and 10 runtime probes reported passing | Recorded checkpoint, not re-executed or independently revalidated here |
| Network-disabled container; read-only root/source; non-root; dropped capabilities; resource bounds | Container restrictions do not establish VM-wide network isolation or general escape resistance |
| Nine-case basic container rehearsal; 12 initial tests; later seven recovery fixtures and 19 expanded tests | Same authored fixtures and repeat runs, not new independent samples |
| Worker replacement/retry fixtures | Not VM restart, power loss, current stop verification or independent watchdog qualification |

The source explicitly records VM-level image-download capability and a writable VM study-folder mount, despite stricter container mounts. Do not collapse the VM and container boundaries.

**Launcher gap:** `run_container.py` calls `subprocess.run` for readiness and execution without a timeout, and captures output without a specified cap. Its restriction settings are useful design precedents; its launcher is **not** our bounded guardian or proof of emergency termination. Do not execute the historical resume commands as part of this review.

Sources: [environment checkpoint](https://github.com/node-and-norm/control-evidence-corpus/blob/f3e8066e2f1db268255914ed0bde08bcd415002b/experiments/us-control-sandbox/ENVIRONMENT.md), [run index](https://github.com/node-and-norm/control-evidence-corpus/blob/f3e8066e2f1db268255914ed0bde08bcd415002b/experiments/us-control-sandbox/RUN-INDEX.md), [launcher source](https://github.com/node-and-norm/control-evidence-corpus/blob/f3e8066e2f1db268255914ed0bde08bcd415002b/experiments/us-control-sandbox/run_container.py). Historical results remain attributed to these records; no archive/hash verification or tests were rerun here.


## 4b. Original dedicated-profile proposal — retained design history

**Original proposal:** choose a new, dedicated profile, proposed name `node-norm-revocation-qual-v01`, rather than starting or repurposing the stopped shared research VM. This proposal itself did not create a profile or authorize startup. A later separately authorized setup is recorded in section 4c; its actual name, resources and shared networking differ from this initial proposal. These are desired controls, not validated Colima/Lima configuration syntax.

| Layer / choice | Concrete proposed setting | Decision or verification still needed |
|---|---|---|
| VM identity and lifecycle | Fresh isolated profile; no autostart or workload restart; do not activate/change the default Docker context | Confirm supported settings, ownership and no unrelated workloads |
| Architecture | Native ARM64 guest/image on the intended Mac; no Rosetta dependency | Verify host/image compatibility without retrying denied reads |
| VM resources | 2 vCPUs, 2 GiB RAM, 10 GiB data-disk budget; record root-disk virtual capacity separately | Availability and host headroom; disk caps are not secure erasure |
| Host mounts | Only dedicated input staging, read-only, and dedicated per-attempt output staging, writable; no home/repository-wide/credential/socket/SSH-agent mounts | Verify effective VM mounts and guest-to-host exposure; scoped writable output is still a host write surface; treat it as untrusted data, never execute it or accept it as authoritative guardian status |
| Guest storage | Fresh disposable data disk for this profile; no old disk reuse; preserve evidence before an approved disposal | Exact retention and deletion decision; no automatic destructive cleanup |
| Setup networking | If dependencies require downloads, separate approved preparation phase with pinned artifacts and explicit destinations | No silent broad outbound access; obtain approval for required network/security changes |
| Test networking | No public/LAN/host egress for fixture containers; no inbound/automatic forwards; no host DNS/proxy inheritance into workload; no registry access during attempts | Verify effective controls at VM and container boundaries; user-mode networking alone is not isolation |
| Management path | Separate narrowly scoped host-to-VM lifecycle channel, inaccessible to the fixture; no general forwarding | Management access must survive workload network isolation; support and security approvals unresolved |
| Fixture container | Pinned image digest; read-only root/source; non-root; all capabilities dropped; no-new-privileges; no network; no engine socket | Verify effective configuration and permissions before any qualification |
| Container budget | 1 CPU, 256 MiB memory, 64 processes; 64 MiB noexec/nosuid/nodev temporary storage | Proposed initial limits only; distinguish resource-limit failures from intended transport faults |
| Evidence budget | One fresh attempt directory; capped status/output (proposed 1 MiB per stream, 10 MiB per attempt); truncation explicitly recorded | Logging must not block stop; incomplete evidence cannot pass |
| Guardian | Host-side owner outside the guest fault domain, not independent of host failure; immutable attempt/boundary identity and absolute deadline | Supported nonblocking lifecycle control, force-stop and all-task exit observation remain unverified |

**Important:** a VM stop command may itself hang; adding a host timeout does not prove the guest stopped. The host guardian must have a verified independent containment/recovery route. Neither the October checkpoint nor today's stopped status supplies that proof.

**Original proposal approval boundary:** this notebook itself approves no creation, startup, dependency installation, mount, forwarding, proxy/DNS, firewall or other security-sensitive change. Before implementation, present the exact version-specific configuration diff, destinations, host paths and effects for the applicable confirmation. Keep the existing VM and its data unchanged.

**Go/no-go sequence:** resolve profile syntax and management controls → review exact security-sensitive changes → obtain required approvals → prepare and verify effective isolation under a separate authorization → qualify watchdog/cleanup → consider the ten-case transport batch. If containment or network isolation cannot be substantiated, stop; do not weaken settings silently.


## 4c. Later authorized setup: dedicated VM is running

**Status update • 12 November 2026, approximately 20:39 UTC. Setup checks recorded, safeguard qualification still open.** The separately authorized actual profile is `revocation-qualification-v01`, not the original proposed name above. This section records the reported setup checks; it does not turn them into a transport or watchdog test.

| Setup item | Reported checked state | Limit |
|---|---|---|
| Dedicated profile | `revocation-qualification-v01` running | Running status is not proof of bounded stop or descendant containment |
| CPU / memory | 2 CPUs and 2048 MiB configured; Docker reports 2 CPUs and 2054496256 bytes | Configuration and reported values, not measured enforcement |
| Storage | 5 GiB root and 2 GiB data | Virtual disk capacities, not current physical disk consumption |
| Host mount | Sole host virtiofs mount is the dedicated staging folder | Private absolute host path omitted; mount isolation is not escape resistance |
| Networking | Shared networking approved for this setup | Not network-disabled operation; isolation requirements for eventual test workloads remain open |
| Existing environment | Old research profile remains stopped | No reuse or restart of that profile is implied |
| Docker runtime / context | Docker server 28.4.0 reports cgroup v2; default context unchanged | Daemon response is not containment/watchdog qualification |
| Host free space | Approximately 18.96 GiB at inspection | Point-in-time observation, not a reservation or guaranteed headroom |
| Workload execution | No test containers, workload images or study tests run in this setup step | Guest/base-image provisioning is distinct from pulling a workload image |

**What changed:** we now have a dedicated running candidate environment. **What did not change:** independent watchdog, force-stop, cleanup, endpoint custody and real-process qualification are still unqualified. No native Temporal Workflow/Activity, model call or measured collection is established by setup.

The network-disabled test-workload proposal is not silently satisfied by approved shared VM networking. Any additional security/network changes retain their approval requirements. A running VM may consume resources; its eventual lifecycle and cleanup must remain explicit.

Sources and scope: [setup verification record](safeguards/VM_SETUP_STATUS.md), supplied by the separately assigned environment reviewer. This notebook update did not start the VM or perform code tests.


## 4d. Why “force-stop succeeded” is not enough

**Source-reviewed limitation, not a newly observed stop failure.** Lima v2.0.3 force-stop sends SIGKILL requests, logs errors and removes temporary metadata without waiting for the original driver/host-agent processes to exit. The CLI then performs network reconciliation with an empty instance selector. An exact stop target does not prove the whole CLI has no shared-network effects; those effects, disk unlocks and metadata removal need review and disclosure before a concrete operation is approved. Neither its return code nor a later `Stopped` label independently proves process exit.

```text
Host guardian (deadline, never waits on lifecycle CLI)
    ├── pre-armed independent host exit observer
    ├── supervised, exact-target lifecycle worker
    └── bounded evidence writer
                   ↓
       dedicated VM / test workload
```

**Proposed readiness:** bind exact profile and stable process identities → register supported OS exit observations → recheck identity/race handling → verify complete process coverage → acknowledge observer ready → only then admit a test attempt. The VM is already running; this is readiness before a future workload, not a claim the observer preceded VM startup.

**Fail closed:** denied or missing identity access, lost registration, unknown process coverage or unqualified stop behavior means **not ready**. Loss of observation after dispatch means **cleanup unresolved**, never a pass. At this design snapshot, process-exit notifications were only a candidate primitive. The later narrowly approved registration result in section 4e advances that prerequisite, but not event delivery or termination qualification.

The host guardian is outside guest failures, but not host failures. It cannot infer authoritative state from guest-writable output, removed PID files or a reused numeric PID. A supported stop request must run outside its deadline loop.

**Decision for this increment:** document the precise design and blockers, not an executable guardian. Prior process-inspection denials remain respected. No lifecycle stop, process-observer test, study test or security change was performed.

[Detailed host guardian design and source links](safeguards/HOST_GUARDIAN_DESIGN.md) · [Lima force-stop source](https://github.com/lima-vm/lima/blob/v2.0.3/pkg/instance/stop.go) · [CLI orchestration](https://github.com/lima-vm/lima/blob/v2.0.3/cmd/limactl/stop.go)


## 4e. Narrow observer readiness: one prerequisite advanced

**12 November 2026 • 21:02:28 UTC. Explicitly authorized targeted check, not a stop test.**

| Step | Reported result | Meaning |
|---|---|---|
| Named VM identity | Owner, executable path and process start time checked against dedicated PID records | Exact observed process binding, not full process-set coverage |
| Native registration | `kqueue` process-exit observer registered successfully | Registration permitted at that moment, not proof of event delivery |
| Identity recheck | Unchanged | No detected identity replacement during this check |
| Immediate poll | Zero events | No exit was induced; this is not a failed or passed delivery test |
| Close | Observer descriptor closed; VM still running | **No observer remains armed** |

The earlier 21:01:54 attempt stopped **before registration** because Python lacked `KQ_EV_RECEIPT`; its descriptor was closed. Standard supported registration flags were then used successfully. This was not a permission denial or bypass. Both attempts are recorded.

**What changed:** the previously unknown targeted identity/registration route has now been demonstrated at a specific time. **Still unproven:** actual exit-event delivery, complete VM coverage, stop/cleanup effectiveness and independent deadline reliability. The earlier broad process-listing denial was not retried; no broad listing, signal, stop, workload or study test ran.

[Detailed readiness record and provenance](safeguards/OBSERVER_READINESS.md). Historical process identifiers must not be reused as live targets. A future observer must be freshly bound and armed under the relevant authorization. Further tests and merge still require checking with Mark.


## 4f. One approved harmless-child notification check

**12 November 2026 • 21:14:16 UTC. PASS for this single normal-child check only.**

```text
Supervisor starts its own harmless child, waiting for one input byte
       → exit observer registered; child confirmed still running
       → supervisor releases child with one byte and closes input
       → matching exit notification in 1.481 seconds (3-second window)
       → child exit code 0; reaped; observer closed
```

The exact reported release-to-event interval was **1.480987333 seconds**. Tool wall time was **7.776 seconds**, including other overhead: the shorter interval is not an end-to-end process-start or hard-deadline guarantee. No fallback kill was needed.

**This adds:** one successful native exit-event delivery for a supervisor-owned, cooperatively exiting child. **It does not add:** VM-stop evidence, stuck-child termination, crash/race coverage, complete process-tree coverage or full guardian reliability.

No VM action, study case or settings change occurred. No persistent observer remains from this check. The source is the separately assigned environment reviewer's tool output, summarized in the [child-check record](safeguards/CHILD_OBSERVER_CHECK.md); no persistent raw artifact was written by the check. Further tests and merge still require checking with Mark.


## 4g. Supervisor source review: corrected, still not ready

A local-only supervisor candidate underwent **separate AI-assisted source review and re-review**. No candidate source was executed, compiled, tested or added to this documentation PR. No VM lifecycle action occurred.

Review corrections addressed configuration checks, late-event classification, release uncertainty, output preservation and cleanup errors. Re-review also closed helper-entry bypass, moved hash checks before invocation, rejected config aliases, tracked real EOF and narrowed completeness terminology. The final source inspection confirmed those changes are present, not that they work at runtime. Both normal and helper entry points remain closed by a development guard, which is not a security boundary.

| Still open | Why it blocks readiness |
|---|---|
| Effective configuration binding | Matching hashes do not prove these are the active profile/network files |
| Identity/replacement races | Files, binaries or process identities can change between checks and use |
| Durable evidence | Final output alone does not preserve all failure paths; persistence cannot block observation |
| Lifecycle-helper descendants | Owned-child cleanup does not establish control of CLI-created helpers or shared-service effects |

**Next smallest step, no test:** write the exact operation-and-evidence contract: command/configuration binding, helper ownership and cleanup map, minimal attempt-record fields, bounded persistence/failure rules and reviewer acceptance criteria. Use source and preserved records; mark unavailable evidence rather than filling gaps with assumptions.

[Review record and proposed next design step](safeguards/SUPERVISOR_SOURCE_REVIEW.md). **No test or merge without checking with Mark.** The dedicated VM's lifecycle is not changed by this documentation.


## 4h. The operation-and-evidence contract

**Proposed contract v0.1, not an implementation or permission to stop the VM.** The [full contract](safeguards/OPERATION_EVIDENCE_CONTRACT.md) turns the four remaining blockers into inspectable requirements.

| Contract component | Required before readiness |
|---|---|
| Effective configuration | Trace installed executable, active profile/network configuration and defaults to their real source; hashes alone are insufficient |
| Identity and races | Fresh lifetime identity and observer recheck; explicit concurrency ownership and residual races |
| Helper ownership | Map CLI subprocesses and shared user-v2 effects; no blanket process-group cleanup |
| Durable evidence | Bounded writer, precise sequence/digest receipts and explicit loss states; never block observation for logs |

```text
Exact operation + approval → active configuration + identity
            → owned actors + pre-armed observer
            → preserved release-attempt record → one release
            → independent observation + CLI result
            → separate cleanup / shared-service / evidence dispositions
```

The plan fixes a 30-second observation window, no force/restart, and separate outcome fields. It specifies preflight, observer-ready, release uncertainty, event retrieval time, CLI streams, cleanup and persistence receipts. A shared-network side effect remains possible; fresh inventory is required.

**Progress:** there is now a concrete specification reviewers can accept or reject. **Not progress claimed:** no blocker is closed merely by documenting it; no implementation, stop, test or merge occurred.

**Smallest next design step:** complete the installed-version helper/side-effect map and effective-configuration resolution chain from source and preserved evidence. Unknown ownership remains a no-go condition. Any test or merge still comes back to Mark first.


## 4i. Important correction: graceful VM stop can force-stop a helper

**Source-reviewed behavior, not a newly executed stop.** Lima v2.0.3's ordinary graceful-stop path can reconcile the shared `user-v2` network: save leases → SIGINT network daemon → poll for roughly five seconds → potentially SIGKILL. It does not independently wait for process exit after that escalation.

**Earlier “no force” wording means no requested forced VM shutdown, not a guarantee that no helper receives SIGKILL.** This matters to the approval question. No shutdown should be run under the broader promise; the possible helper action must be disclosed and the exact operation approved or replaced by a separately reviewed supported option.

| Source finding | Consequence |
|---|---|
| `AutoStartedIdentifier` can select `launchctl bootout` instead of direct host-agent SIGINT | Active branch remains unverified; do not assume launchd is absent |
| Graceful path has 3m10s event wait, then up to 3m status wait | Our 30s is an observation budget, not Lima's built-in deadline |
| Reconciliation may launch detached usernet, write logs/leases, or stop shared service | CLI parentage/process groups do not establish helper ownership |
| Direct limactl skips Colima daemon-stop and host-address cleanup steps | VM-process exit is not complete Colima cleanup |
| `LIMA_HOME` resolves configuration; missing networks file may create defaults, and missing user-v2 may be filled | Bind existing effective config; loading defaults is not inherently read-only |

Previously reported preflight found the dedicated VM the only running user-v2 consumer; if unchanged, its stop may stop that service. This is an inference, not a new runtime result. Existing forwarding means the setup is not network-isolated.

[Full source map, links and active-state gaps](safeguards/LIFECYCLE_SOURCE_MAP.md). Keep VM process, CLI, Colima helpers and shared-network outcomes separate. All guardian gates remain open; no VM action, test, security change or merge occurred.


## 5. What the critical review changed

| Tempting shortcut | Review finding | Required response |
|---|---|---|
| Kill a process group | Descendants can leave the group; scans race | Prefer a managed boundary; otherwise explicitly narrow to cooperative fixtures |
| Track a pidfd | Identifies one process, not a descendant tree | Do not infer group-wide identity or safe reuse of a numeric PGID |
| Start timer before `Popen` | Guardian can block during launch itself | Establish guardian/boundary first; move blocking launch inside supervision |
| Trust a stop acknowledgment | Signal delivery is not observed exit | Independently verify all in-scope tasks ended |
| Save all logs before stopping | Persistence can hang | Bound status and logging; never delay emergency containment for evidence |
| Weak-reference endpoint claim | Covers the same Python object only | Account for every descriptor duplicate; prohibit fresh-session retry |

The review was separately assigned **AI-assisted design review**, not independent human validation. Its conclusion was conditional design acceptance, not readiness to run.


## 6. Changes required before the real-process gate

- Add an independently supervised launcher and guardian with a documented failure model.
- Establish actual containment membership, ownership and a separately bounded stop path.
- Replace the historical controller's possibly blocking stop RPC and unbounded post-kill wait; do not silently inherit them into this harness.
- Record every intended endpoint duplicate and its lifetime. The worker may need a duplicate, so do not claim the guardian has sole physical ownership.
- Use fresh, unique attempt folders and permanently discard uncertain transports. No retry after possible send.
- Keep transport outcome, evidence completeness and cleanup result separate. `reply_observed` alone cannot qualify a case.
- Keep runtime exporters and the four-domain observation cutoff as later integration work. The existing `ObservationCut` trusts declared acknowledgments and does not prove an atomic snapshot.

These changes are **not implemented by this notebook**. [Observation-cut source](observation_cut.py) · [Candidate limitations](README.md).


## 7. Planned smallest real-process batch

**Not authorized to run merely by reading this notebook.** Preconditions: reviewed implementation, suitable permitted environment, verified safeguard capability and explicit execution approval. Inert synthetic fixtures only; no Temporal service, model/API calls or production actions.

| ID | Deliberate condition | Required observation / expected classification |
|---|---|---|
| T01 | Timely well-formed reply | Exactly one gateway receipt; bound reply; worker exit; complete records and cleanup |
| T02 | Peer never reads; send blocks | Actual blocked-send condition established; deadline containment; uncertain; no replay |
| T03 | Peer receives but never replies | Independent receipt retained; uncertain response; contained process |
| T04 | Reply frame starts but never completes | Partial-frame receipt established; readable is not complete; bounded containment |
| T05 | Peer closes during exchange | EOF/error retained; no retry; receipt evidence determines what is known |
| T06 | Valid reply clearly after cutoff | Never timely success; late evidence retained separately |
| T07 | Wrong request ID in saved result envelope, introduced by a reviewed fixture | No accepted transport success; misbound envelope preserved. This does not test a peer-supplied ID inside the opaque gateway payload. |
| T08 | Oversized reply | No accepted transport success; bounded handling and evidence |
| T09 | Intent-record write fails before send | Failure observed; zero gateway receipts within complete fixture observation; missing journal alone proves nothing |
| T10 | Child resists ordinary termination | External containment exercised; candidate cleanup failure remains visible even if outer recovery succeeds |

**Binding limitation:** the worker builds the outer result-envelope ID from the original request; the parent checks that envelope, while `gateway_result` is opaque. Peer-level response identity validation is not implemented by this check and needs a separate schema/design decision.

One attempt per declared scenario, maximum ten attempts. Fault induction must itself be reviewed and shown to have occurred; do not alter the production fixed request merely to force a pipe condition. If the current wire shape cannot safely exercise a proposed fault, revise that fixture plan before the batch rather than substitute an easier test and call it passed.

Deadline races, descriptor-alias enforcement, crash durability and hostile escape tests are outside this initial batch. Ten controls do not estimate operational reliability.


## 8. Evidence that must survive each attempt

**Identity:** source digest, environment/version, exact command/input, attempt ID and frozen budgets.

**Independent observations:** process/boundary identity, creation and exit observations, watchdog timing, gateway receipt count and complete synthetic request/response records.

**Candidate records:** dispatch intent, reply, worker error and supervisor record, including explicit missing/late status. Preserve primary and cleanup failures separately.

**Provenance:** immutable attempt directory, artifact digests and readable case table. Digests establish byte identity, not truthful or complete records.

There is no native history in this transport-only batch. Do not manufacture four-domain completeness. Termination is cleanup evidence, not proof that cancellation stopped an Activity or that no downstream effect occurred.


## 9. Pass, fail, uncertain — and when to stop

| Result | Meaning |
|---|---|
| PASS (control) | Intended fault occurred; expected conservative classification, bounded dispatch, containment and evidence requirements were met |
| FAIL (control) | Duplicate dispatch, false timely success, accepted misbound reply, tainted-channel reuse or exceeded containment contract |
| UNCERTAIN / NOT QUALIFIED | Fault not established, evidence missing/conflicting or process state unresolved |

A transport result of **uncertain** can be the correct result of a passing fault-control test. It is not evidence of no effect. Candidate cleanup and outer recovery must be reported separately.

**Stop the batch immediately** on permission denial, unexpected external access, duplicate dispatch, uncontrolled descendants, watchdog failure or unreliable observation. Preserve the attempt. No automatic retries or denominator removal. A corrective rerun needs a revised recorded plan.


## 10. Team responsibilities and AI disclosure

| Responsibility | Assignment / authority |
|---|---|
| Research direction and consequential approvals | Mark, human study lead |
| Coordination, source traceability and readable report | Alfred, AI-assisted coordinator |
| Implementation work, if separately approved | Assigned engineering role; source and test changes recorded |
| Critical review | Separately assigned reviewer; record AI assistance and unresolved findings |
| Environment ownership / recovery | Must be identified before execution; not presumed available |

SORA, MIRA and TRACE are part of the user's team vocabulary; this chapter does not invent particular assignments or imply they are persistent live agents. [Team charter](../../docs/team/CHARTER.md) supplies the role basis.

This chapter was assembled with AI assistance from source files, tool observations and separate AI-assisted critique. Design approval is not execution approval or empirical validation. It does not disclose complete private research conversations or claim such a transcript exists.

Jupyter provides the readable document format. The LangChain-inspired progression is **scope → evidence → review → synthesis**; no live LangGraph orchestration is added here.


## 11. The next decision and definition of done

```text
Current: design + metadata review
      ↓
Documented suitable containment capability
      ↓
Reviewed implementation + explicit execution approval
      ↓
Bounded real-process transport qualification report
      ↓
Separate native Temporal integration / qualification decision
      ↓
Separate measured-collection decision
```

**Today:** retain draft PR #7 as experimental. Do not run real-process qualification in the inspected runtime under a guaranteed-stop claim.

**This gate is complete only when** the exact candidate/environment has a reviewed transport-control report including unsuccessful and unresolved cases. That report is not native Temporal qualification, study completion, compliance certification or independent human validation.

If stronger containment cannot be established, bring an explicit weaker trusted-fixture proposal for review. Do not silently downgrade the safeguards.

[Back to candidate](README.md) · [Research-book contents](../../notebooks/00_start_here.ipynb) · [Study report](../../RESEARCH_STUDY.ipynb)
